In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 3 - WEEK 10 BAYESIAN OPTIMISATION
# Run from inside the week10/ folder
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 9.
# - Check Week 9 calibration.
# - Search around the best ACTUALLY observed point.
# - Use a tighter ARD + empirical trust region.
# - Do not automatically expand if acquisition hits boundary.
# ============================================================


# ------------------------------------------------------------
# 1. Load Week 10 cumulative data
# ------------------------------------------------------------

X = np.load("function3/initial_inputs.npy")
Y = np.load("function3/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. WEEK 9 CALIBRATION CHECK
# ------------------------------------------------------------
#
# Week 9 selected:
# [0.320249, 0.604961, 0.415230]
#
# Week 9 chosen from beta=0.1 UCB:
# predicted mean ≈ -0.002006
# predicted std  ≈ 0.002150
#
# Actual:
# -0.025179330694814796
# ------------------------------------------------------------

week9_pred_mean = -0.002006
week9_pred_std = 0.002150
week9_actual = -0.025179330694814796

week9_error = (
    week9_actual
    - week9_pred_mean
)

week9_z_error = (
    week9_error
    / week9_pred_std
)

print("\n================================")
print("WEEK 9 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week9_pred_mean)
print("Predicted std :", week9_pred_std)
print("Actual        :", week9_actual)

print("\nPrediction error:")
print(week9_error)

print("\nError / predicted std:")
print(week9_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(3) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = (
    inverse_ls / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Empirical local scale around incumbent
# ------------------------------------------------------------

other_mask = np.arange(len(X)) != best_idx

distances_to_best = np.linalg.norm(
    X[other_mask] - best_x,
    axis=1
)

nearest_distance = distances_to_best.min()

# More conservative than Week 9 because calibration failed.
empirical_cap = min(
    1.25 * nearest_distance,
    0.08
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print("Nearest point to current best:")
print(nearest_distance)

print("\nEmpirical cap:")
print(empirical_cap)


# ------------------------------------------------------------
# 6. Conservative ARD trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.20 * lengthscales,
    0.015,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("\n================================")
print("WEEK 10 TRUST REGION")
print("================================")

print("Centre:")
print(best_x)

print("\nHalf-widths:")
print(trust_half_width)

print("\nLower:")
print(lower)

print("\nUpper:")
print(upper)


# ------------------------------------------------------------
# 7. Trust-region candidate pool
# ------------------------------------------------------------

rng = np.random.default_rng(42)

tr_candidates = rng.uniform(
    lower,
    upper,
    size=(300000, 3)
)

tree = cKDTree(X)

distance, _ = tree.query(
    tr_candidates,
    k=1
)

tr_candidates = tr_candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(tr_candidates))


# ------------------------------------------------------------
# 8. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    tr_candidates,
    return_std=True
)


# ------------------------------------------------------------
# 9. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI - TRUST REGION")
print("================================")

print("candidate =", tr_candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 10. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", tr_candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 11. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", tr_candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 12. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", tr_candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 13. Distance from incumbent
# ------------------------------------------------------------

def distance_from_best(x):
    return np.linalg.norm(
        x - best_x
    )

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    distance_from_best(
        tr_candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    distance_from_best(
        tr_candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        distance_from_best(
            tr_candidates[idx]
        )
    )


# ------------------------------------------------------------
# 14. Boundary diagnostic
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(len(x)):

        if abs(x[j] - lower[j]) <= tol:
            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(x[j] - upper[j]) <= tol:
            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        tr_candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        tr_candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        boundary_status(
            tr_candidates[idx],
            lower,
            upper
        )
    )

DATA
X shape: (24, 3)
Y shape: (24,)

Current best:
[0.332348 0.553987 0.420645] -> -0.00263018431711505

Y range:
min = -0.3989255131463011
max = -0.00263018431711505
std = 0.07667685894580349

WEEK 9 CALIBRATION CHECK
Predicted mean: -0.002006
Predicted std : 0.00215
Actual        : -0.025179330694814796

Prediction error:
-0.023173330694814795

Error / predicted std:
-10.778293346425485

GP FIT

Fitted kernel:
1.73**2 * Matern(length_scale=[0.792, 1.28, 0.266], nu=2.5) + WhiteKernel(noise_level=0.0158)

ARD lengthscales:
[0.79200545 1.2827184  0.26634176]

Normalised inverse-lengthscale sensitivity:
[0.21781337 0.13448733 0.6476993 ]

EMPIRICAL LOCAL SCALE
Nearest point to current best:
0.05266931461486845

Empirical cap:
0.06583664326858556

WEEK 10 TRUST REGION
Centre:
[0.332348 0.553987 0.420645]

Half-widths:
[0.06583664 0.06583664 0.05326835]

Lower:
[0.26651136 0.48815036 0.36737665]

Upper:
[0.39818464 0.61982364 0.47391335]

Candidates after duplicate filtering:
298688

PRIM

In [2]:
# ============================================================
# FUNCTION 3 WEEK 10 - TIGHT CALIBRATION-AWARE TRUST REGION
# ============================================================
#
# Week 9 missed by -10.78 predictive standard deviations.
#
# The first Week 10 trust-region search sends every
# acquisition to the x1 upper boundary, but none predicts a
# mean better than the actually observed incumbent.
#
# Therefore:
# - tighten around the actual best observation
# - retain ARD information
# - do NOT expand if a boundary is hit again
# ============================================================


# ------------------------------------------------------------
# 1. Tighten existing trust-region widths by 50%
# ------------------------------------------------------------

tight_half_width = np.maximum(
    0.5 * trust_half_width,
    0.012
)

tight_lower = np.maximum(
    0.0,
    best_x - tight_half_width
)

tight_upper = np.minimum(
    1.0,
    best_x + tight_half_width
)

print("================================")
print("TIGHT TRUST REGION")
print("================================")

print("Centre:")
print(best_x)

print("\nHalf-widths:")
print(tight_half_width)

print("\nLower:")
print(tight_lower)

print("\nUpper:")
print(tight_upper)


# ------------------------------------------------------------
# 2. Dense stochastic search
# ------------------------------------------------------------

rng_tight = np.random.default_rng(123)

tight_candidates = rng_tight.uniform(
    tight_lower,
    tight_upper,
    size=(350000, 3)
)

distance, _ = tree.query(
    tight_candidates,
    k=1
)

tight_candidates = tight_candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(tight_candidates))


# ------------------------------------------------------------
# 3. Predictions
# ------------------------------------------------------------

tight_mu, tight_sigma = gp.predict(
    tight_candidates,
    return_std=True
)


# ------------------------------------------------------------
# 4. EI
# ------------------------------------------------------------

tight_EI = expected_improvement(
    tight_mu,
    tight_sigma,
    best_y,
    xi=0.0
)

tight_ei_idx = np.argmax(tight_EI)

print("\n================================")
print("TIGHT EI")
print("================================")

print(
    "candidate =",
    tight_candidates[tight_ei_idx]
)

print(
    "mean =",
    tight_mu[tight_ei_idx]
)

print(
    "std =",
    tight_sigma[tight_ei_idx]
)

print(
    "EI =",
    tight_EI[tight_ei_idx]
)


# ------------------------------------------------------------
# 5. Highest mean
# ------------------------------------------------------------

tight_mean_idx = np.argmax(tight_mu)

print("\n================================")
print("TIGHT HIGHEST MEAN")
print("================================")

print(
    "candidate =",
    tight_candidates[tight_mean_idx]
)

print(
    "mean =",
    tight_mu[tight_mean_idx]
)

print(
    "std =",
    tight_sigma[tight_mean_idx]
)


# ------------------------------------------------------------
# 6. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("TIGHT UCB")
print("================================\n")

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    tight_UCB = (
        tight_mu
        + beta * tight_sigma
    )

    idx = np.argmax(tight_UCB)

    print(
        f"beta={beta}",
        "\n candidate =",
        tight_candidates[idx],
        "\n mean =",
        round(tight_mu[idx], 6),
        "\n std =",
        round(tight_sigma[idx], 6),
        "\n UCB =",
        round(tight_UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 7. Distance from actual incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        tight_candidates[tight_ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        tight_candidates[tight_mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    tight_UCB = (
        tight_mu
        + beta * tight_sigma
    )

    idx = np.argmax(tight_UCB)

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            tight_candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 8. Boundary check
# ------------------------------------------------------------

print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        tight_candidates[tight_ei_idx],
        tight_lower,
        tight_upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        tight_candidates[tight_mean_idx],
        tight_lower,
        tight_upper
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    tight_UCB = (
        tight_mu
        + beta * tight_sigma
    )

    idx = np.argmax(tight_UCB)

    print(
        f"UCB beta={beta}:",
        boundary_status(
            tight_candidates[idx],
            tight_lower,
            tight_upper
        )
    )

TIGHT TRUST REGION
Centre:
[0.332348 0.553987 0.420645]

Half-widths:
[0.03291832 0.03291832 0.02663418]

Lower:
[0.29942968 0.52106868 0.39401082]

Upper:
[0.36526632 0.58690532 0.44727918]

Candidates after duplicate filtering:
343610

TIGHT EI
candidate = [0.36511472 0.57220565 0.44727523]
mean = -0.011706950739222746
std = 0.0118153991349691
EI = 0.0015016029732496956

TIGHT HIGHEST MEAN
candidate = [0.36520441 0.5554085  0.43974572]
mean = -0.011575031459667956
std = 0.011441108629506683

TIGHT UCB

beta=0.05 
 candidate = [0.36526491 0.55724409 0.4396284 ] 
 mean = -0.011575 
 std = 0.011446 
 UCB = -0.011003 

beta=0.1 
 candidate = [0.36521519 0.55559317 0.44303879] 
 mean = -0.011584 
 std = 0.011555 
 UCB = -0.010428 

beta=0.25 
 candidate = [0.36521519 0.55559317 0.44303879] 
 mean = -0.011584 
 std = 0.011555 
 UCB = -0.008695 

beta=0.5 
 candidate = [0.36522486 0.56084735 0.44709609] 
 mean = -0.011644 
 std = 0.011738 
 UCB = -0.005775 

beta=1.0 
 candidate = [0.365114

In [3]:
# ============================================================
# FINAL FUNCTION 3 - WEEK 10 SELECTION
# ============================================================
#
# Week 9 produced a severe calibration miss (~ -10.78 sigma).
#
# A first conservative trust region and a second 50%-contracted
# trust region both showed a persistent directional preference
# toward increasing x1.
#
# We do NOT expand further because recent calibration was poor.
#
# Within the tightly controlled region, choose the highest
# posterior mean rather than EI/UCB exploration.
#
# This is the most exploitative supported candidate and avoids
# additionally pushing x3 to its trust-region boundary.

final_idx = np.argmax(tight_mu)

week10_candidate = tight_candidates[final_idx]

print("Week 10 Function 3 candidate:")
print(week10_candidate)

print("\nPredicted mean:")
print(tight_mu[final_idx])

print("\nPredicted std:")
print(tight_sigma[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week10_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week10_candidate
)

print("\nPortal format:")
print(portal)

Week 10 Function 3 candidate:
[0.36520441 0.5554085  0.43974572]

Predicted mean:
-0.011575031459667956

Predicted std:
0.011441108629506683

Distance from current best:
0.0380315844945704

Portal format:
0.365204-0.555408-0.439746
